# Stage 07 — Evaluate models on the test split

| | |
|---|---|
| **Intention** | Score BLEU / chrF / gloss-token F1 and write `artifacts/comparison.json`. |
| **Input** | `data/mbart/test.jsonl` + `artifacts/ckpts/*/best` |
| **Output** | `artifacts/comparison.json`, `artifacts/predictions/*_test.jsonl` |
| **Runtime** | minutes–tens of minutes depending on which models exist |


In [1]:
import sys
from pathlib import Path

ROOT = Path("..").resolve()
sys.path.insert(0, str(ROOT / "src"))

from ttg.config import DATA_DIR, CHECKPOINTS_DIR, ARTIFACTS, PROJECT_ROOT
print("PROJECT_ROOT =", PROJECT_ROOT)
print("DATA_DIR     =", DATA_DIR)
print("CHECKPOINTS  =", CHECKPOINTS_DIR)


PROJECT_ROOT = /home/khurshida/Projects/uzsl-text-to-gloss
DATA_DIR     = /home/khurshida/Projects/uzsl-text-to-gloss/data
CHECKPOINTS  = /home/khurshida/Projects/uzsl-text-to-gloss/artifacts/ckpts


In [ ]:
# Restrict to one model while iterating, or "all".
ONLY = "nllb"   # sockeye | mbart | nllb | qwen | gemma | all
# "gloss2text" scores the "*_g2t" checkpoints (gloss in, sentence out) on the
# same 130-row held-out split, just with input/reference swapped — sockeye
# has no gloss2text checkpoint (never trained either direction) and is
# skipped automatically for this direction.
DIRECTION = "text2gloss"   # "text2gloss" or "gloss2text"


In [ ]:
import json
from ttg.eval import evaluate

results = evaluate(only=ONLY, direction=DIRECTION)
print(json.dumps(results, indent=2))


## Comparison

`comparison.json` only holds whatever was scored on the *last* run of each model — running with
`ONLY = "nllb"` above just refreshes the `nllb200` entry and leaves the others as they were. `evaluate()`
now also drops any cached entry whose checkpoint no longer exists under `artifacts/ckpts/` (e.g. a
`sockeye3` entry from an old run once its checkpoint was deleted), so stale/abandoned models don't linger
in the comparison forever.

Each model below was still scored on a different slice of the 140-example test set (see `n`) —
`mbart50`/`qwen25` on the first 20, `gemma2` on 98, and only `nllb200` on the full 140 — so this is not
yet an apples-to-apples comparison. Re-run with `ONLY = "all"` (all four checkpoints present) to score
every model on the same 140 rows.

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

comparison_file = "comparison_gloss2text.json" if DIRECTION == "gloss2text" else "comparison.json"
comparison = json.loads((ARTIFACTS / comparison_file).read_text(encoding="utf-8"))

df = pd.DataFrame(comparison["models"]).T
df = df.rename(columns={"num_examples": "n", "gloss_token_f1": "gloss_f1", "exact_match": "exact"})
df = df[["n", "bleu", "chrf", "gloss_f1", "exact"]]
display(df.round(3))

metrics = ["bleu", "chrf", "gloss_f1", "exact"]
fig, axes = plt.subplots(1, len(metrics), figsize=(16, 3.5))
for ax, metric in zip(axes, metrics):
    ax.bar(df.index, df[metric], color=["#2a78d6", "#eb6834", "#1baf7a", "#eda100"][: len(df)])
    ax.set_title(metric)
    ax.tick_params(axis="x", rotation=45)
fig.tight_layout()
plt.show()
